In [0]:
%sql
CREATE or replace table oms.oms.gym_entry (
    entry_id INT PRIMARY KEY,
    member_id INT,
    check_in_time TIMESTAMP,
    check_out_time TIMESTAMP,
    last_updated TIMESTAMP
);

INSERT INTO oms.oms.gym_entry (entry_id, member_id, check_in_time, check_out_time, last_updated) VALUES
(1, 101, '2025-06-30 08:00:00', '2025-06-30 09:00:00', '2025-06-30 09:00:00'),
(2, 102, '2025-06-30 08:15:00', '2025-06-30 09:10:00', '2025-06-30 09:10:00'),
(3, 103, '2025-06-30 09:00:00', '2025-06-30 10:00:00', '2025-06-30 10:00:00');



In [0]:
sheet_1 = spark.sql("select * from oms.oms.gym_entry")
sheet_1.display()

In [0]:
from pyspark.sql.functions import when, col

sheet_1 = sheet_1.withColumn(
    "Fees",
    when(col("entry_id") == 1, 8000)
    .when(col("entry_id") == 2, 10000)
    .when(col("entry_id") == 3, 12000)
    .when(col("entry_id") == 4, 15000)
    .when(col("entry_id") == 5, 18000)
    .otherwise(0)  # default value if no match
)


In [0]:
sheet_1.display()

In [0]:
%sql
crate table oms.oms.gym_update

In [0]:
from delta.tables import DeltaTable

# Load the Delta table
delta_table = DeltaTable.forName(spark, "oms.oms.gym_entry")

# Prepare the updates DataFrame
updates_df = spark.createDataFrame([
    (2, 102, "2025-06-30T08:15:00.000+00:00", "2025-06-30T09:10:00.000+00:00", "2025-07-01T09:10:00.000+00:00", 15000)
], ["entry_id", "member_id", "check_in_time", "check_out_time", "last_updated", "Fees"])

# Perform the merge (SCD Type 1 style update)
delta_table.alias("target").merge(
    updates_df.alias("updates_df"),
    "target.entry_id = updates_df.entry_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()


In [0]:
updates_df = delta_table
updates_df.display()